# PromptChef Colab 튜토리얼

이 노트북은 Colab 환경에서 [PromptChef](./README.md)의 규칙 기반 파이프라인을 바로 실행해 볼 수 있도록 최소한의 예제를 제공합니다. FastAPI 서버를 띄우지 않고도 핵심 함수를 호출해 프롬프트 플랜·초안·평가 결과를 확인할 수 있습니다.


## 1. 환경 설정

Colab에서 깃허브 저장소를 열면 작업 디렉터리에 이미 소스가 포함되어 있습니다. 프로젝트 루트에서 의존성을 설치하고 패키지를 로컬 모드로 등록합니다.


In [ ]:
import sys, os
print(sys.version)
print('작업 디렉터리:', os.getcwd())

!pip install -e .


## 2. 기본 파이프라인 실행 (`compose_and_run`)

`ComposeAndRunRequest` 모델에 프로필과 사용자 입력을 전달하면 Planner → Composer → Runner → Evaluator → Refiner 단계를 한 번에 수행합니다. 반환 값에는 플랜, 합성된 프롬프트, 초안(preview), 보정본(final_output), 토큰/지연 메타 정보가 포함됩니다.


In [ ]:
from app.models import ComposeAndRunRequest
from app.services.pipeline_service import compose_and_run

request = ComposeAndRunRequest(
    profile={
        "role": "마케팅 매니저",
        "today_goal": "주간 보고",
        "tone_pref": "formal",
        "interests": ["성과 지표", "캠페인"],
    },
    user_input="회의 메모: 신규 캠페인 CTR 12% 개선, 예산 5% 상향 요청",
)

response = compose_and_run(request)
response


## 3. 응답 필드 살펴보기

필요한 필드만 선택해서 살펴보면 플랜과 생성 결과의 구성을 쉽게 이해할 수 있습니다.


In [ ]:
print("== 플랜 ==")
print(response.plan.model_dump())

print("\n== 시스템/유저 프롬프트 ==")
print(response.bundle.model_dump())

print("\n== 초안(preview) ==\n", response.preview)
print("\n== 보정본(final_output) ==\n", response.final_output)
print("\n== 메타 ==\n", response.meta.model_dump())


## 4. 자동 반복 (`auto_compose_and_run`)

초안 평가 점수가 목표치에 도달할 때까지 플랜에 제약을 추가하며 자동으로 재시도할 수 있습니다. 기본 설정으로는 최대 3회 시도하며, `target_score`(기본 0.85)를 넘으면 중단합니다.


In [ ]:
from app.services.pipeline_service import auto_compose_and_run

auto_response = auto_compose_and_run(request, max_rounds=3, target_score=0.9)
print("재시도 횟수:", len(auto_response.evaluations))
print("최종 점수:", auto_response.evaluations[-1].score)

print("\n== 제약 조건 변화 ==")
for idx, plan in enumerate([auto_response.final.plan], start=1):
    print(f"Round {idx}: {plan.constraints}")

print("\n== 최종 보정본 ==\n", auto_response.final.final_output)


## 5. FastAPI 서버 실행 (선택)

실제 API 엔드포인트로 호출하고 싶다면 Uvicorn을 백그라운드로 띄운 뒤 `requests`로 호출할 수 있습니다. Colab에서는 노트북 런타임을 유지하는 동안에만 서버가 동작합니다.

```python
import nest_asyncio, uvicorn, threading
from app.main import app

nest_asyncio.apply()

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=run_server, daemon=True).start()
```

서버가 뜬 뒤에는 다음과 같이 호출합니다.

```python
import requests
payload = request.model_dump()
requests.post("http://localhost:8000/compose_and_run", json=payload).json()
```
